In [1]:
import os
os.environ["PYSPARK_PYTHON"] = r"C:\Users\krish\anaconda3\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\krish\anaconda3\python.exe"

In [2]:
from pyspark.context import SparkContext
from pyspark.sql.session import SparkSession
from pyspark.ml.feature import StringIndexer, VectorAssembler, VectorAssembler, StandardScaler
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.sql import functions as F
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

In [4]:
# Create Spark context and session (for local execution)
sc = SparkContext.getOrCreate()
spark = SparkSession(sc)

In [5]:
print("Display Spark context information (version, master, app name, and UI link)")
sc

Display Spark context information (version, master, app name, and UI link)


<SparkContext master=local[*] appName=pyspark-shell>

In [6]:
spark

In [7]:
# File location and type
file_location = "../Data/Mumbai House Price.csv"
file_type = "csv"

# CSV options
infer_schema = "True"
first_row_is_header = "True"
delimiter = ","

# The applied options are for CSV files. For other file types, these will be ignored.
df = spark.read \
            .format(file_type) \
            .option("inferSchema", infer_schema) \
            .option("header", first_row_is_header) \
            .option("sep", delimiter) \
            .load(file_location)

df

DataFrame[Price: int, Area: int, Location: string, No. of Bedrooms: int, Resale: int, MaintenanceStaff: int, Gymnasium: int, SwimmingPool: int, LandscapedGardens: int, JoggingTrack: int, RainWaterHarvesting: int, IndoorGames: int, ShoppingMall: int, Intercom: int, SportsFacility: int, ATM: int, ClubHouse: int, School: int, 24X7Security: int, PowerBackup: int, CarParking: int, StaffQuarter: int, Cafeteria: int, MultipurposeRoom: int, Hospital: int, WashingMachine: int, Gasconnection: int, AC: int, Wifi: int, Children'splayarea: int, LiftAvailable: int, BED: int, VaastuCompliant: int, Microwave: int, GolfCourse: int, TV: int, DiningTable: int, Sofa: int, Wardrobe: int, Refrigerator: int]

In [8]:
print("Print schema to understand data types and structure of the loaded dataset")
df.printSchema()

Print schema to understand data types and structure of the loaded dataset
root
 |-- Price: integer (nullable = true)
 |-- Area: integer (nullable = true)
 |-- Location: string (nullable = true)
 |-- No. of Bedrooms: integer (nullable = true)
 |-- Resale: integer (nullable = true)
 |-- MaintenanceStaff: integer (nullable = true)
 |-- Gymnasium: integer (nullable = true)
 |-- SwimmingPool: integer (nullable = true)
 |-- LandscapedGardens: integer (nullable = true)
 |-- JoggingTrack: integer (nullable = true)
 |-- RainWaterHarvesting: integer (nullable = true)
 |-- IndoorGames: integer (nullable = true)
 |-- ShoppingMall: integer (nullable = true)
 |-- Intercom: integer (nullable = true)
 |-- SportsFacility: integer (nullable = true)
 |-- ATM: integer (nullable = true)
 |-- ClubHouse: integer (nullable = true)
 |-- School: integer (nullable = true)
 |-- 24X7Security: integer (nullable = true)
 |-- PowerBackup: integer (nullable = true)
 |-- CarParking: integer (nullable = true)
 |-- Staff

##### Note on output readability due to wide feature set

In [9]:
print("The output to show function will not be beautiful because of the number of features in dataset.")
df.show(5)

The output to show function will not be beautiful because of the number of features in dataset.
+-------+----+--------+---------------+------+----------------+---------+------------+-----------------+------------+-------------------+-----------+------------+--------+--------------+---+---------+------+------------+-----------+----------+------------+---------+----------------+--------+--------------+-------------+---+----+------------------+-------------+---+---------------+---------+----------+---+-----------+----+--------+------------+
|  Price|Area|Location|No. of Bedrooms|Resale|MaintenanceStaff|Gymnasium|SwimmingPool|LandscapedGardens|JoggingTrack|RainWaterHarvesting|IndoorGames|ShoppingMall|Intercom|SportsFacility|ATM|ClubHouse|School|24X7Security|PowerBackup|CarParking|StaffQuarter|Cafeteria|MultipurposeRoom|Hospital|WashingMachine|Gasconnection| AC|Wifi|Children'splayarea|LiftAvailable|BED|VaastuCompliant|Microwave|GolfCourse| TV|DiningTable|Sofa|Wardrobe|Refrigerator|
+-------

In [10]:
print("Features of RAW dataset:\n", df.columns)

Features of RAW dataset:
 ['Price', 'Area', 'Location', 'No. of Bedrooms', 'Resale', 'MaintenanceStaff', 'Gymnasium', 'SwimmingPool', 'LandscapedGardens', 'JoggingTrack', 'RainWaterHarvesting', 'IndoorGames', 'ShoppingMall', 'Intercom', 'SportsFacility', 'ATM', 'ClubHouse', 'School', '24X7Security', 'PowerBackup', 'CarParking', 'StaffQuarter', 'Cafeteria', 'MultipurposeRoom', 'Hospital', 'WashingMachine', 'Gasconnection', 'AC', 'Wifi', "Children'splayarea", 'LiftAvailable', 'BED', 'VaastuCompliant', 'Microwave', 'GolfCourse', 'TV', 'DiningTable', 'Sofa', 'Wardrobe', 'Refrigerator']


In [11]:
 df = df.drop('No. of Bedrooms')

In [12]:
print("Features of dataset after dropping 'No. of Bedrooms':\n", df.columns)

Features of dataset after dropping 'No. of Bedrooms':
 ['Price', 'Area', 'Location', 'Resale', 'MaintenanceStaff', 'Gymnasium', 'SwimmingPool', 'LandscapedGardens', 'JoggingTrack', 'RainWaterHarvesting', 'IndoorGames', 'ShoppingMall', 'Intercom', 'SportsFacility', 'ATM', 'ClubHouse', 'School', '24X7Security', 'PowerBackup', 'CarParking', 'StaffQuarter', 'Cafeteria', 'MultipurposeRoom', 'Hospital', 'WashingMachine', 'Gasconnection', 'AC', 'Wifi', "Children'splayarea", 'LiftAvailable', 'BED', 'VaastuCompliant', 'Microwave', 'GolfCourse', 'TV', 'DiningTable', 'Sofa', 'Wardrobe', 'Refrigerator']


### Checking for Null values

In [13]:
print("Check for missing values across all columns")
df.select([F.count(F.when(F.col(feature).isNull(), feature)) \
           .alias(feature) \
           for feature in df.columns]) \
    .show()

Check for missing values across all columns
+-----+----+--------+------+----------------+---------+------------+-----------------+------------+-------------------+-----------+------------+--------+--------------+---+---------+------+------------+-----------+----------+------------+---------+----------------+--------+--------------+-------------+---+----+------------------+-------------+---+---------------+---------+----------+---+-----------+----+--------+------------+
|Price|Area|Location|Resale|MaintenanceStaff|Gymnasium|SwimmingPool|LandscapedGardens|JoggingTrack|RainWaterHarvesting|IndoorGames|ShoppingMall|Intercom|SportsFacility|ATM|ClubHouse|School|24X7Security|PowerBackup|CarParking|StaffQuarter|Cafeteria|MultipurposeRoom|Hospital|WashingMachine|Gasconnection| AC|Wifi|Children'splayarea|LiftAvailable|BED|VaastuCompliant|Microwave|GolfCourse| TV|DiningTable|Sofa|Wardrobe|Refrigerator|
+-----+----+--------+------+----------------+---------+------------+-----------------+----------

### Encoding using String Indexer

In [14]:
print("Encode categorical 'Location' column using StringIndexer")
indexer_model = StringIndexer(inputCols = ["Location"], 
                              outputCols =["Location_indexed"]).fit(df)
df_indexed = indexer_model.transform(df)
df_indexed = df_indexed.drop("Location")

Encode categorical 'Location' column using StringIndexer


In [15]:
print("Check schema after string indexing")
df_indexed.printSchema()

Check schema after string indexing
root
 |-- Price: integer (nullable = true)
 |-- Area: integer (nullable = true)
 |-- Resale: integer (nullable = true)
 |-- MaintenanceStaff: integer (nullable = true)
 |-- Gymnasium: integer (nullable = true)
 |-- SwimmingPool: integer (nullable = true)
 |-- LandscapedGardens: integer (nullable = true)
 |-- JoggingTrack: integer (nullable = true)
 |-- RainWaterHarvesting: integer (nullable = true)
 |-- IndoorGames: integer (nullable = true)
 |-- ShoppingMall: integer (nullable = true)
 |-- Intercom: integer (nullable = true)
 |-- SportsFacility: integer (nullable = true)
 |-- ATM: integer (nullable = true)
 |-- ClubHouse: integer (nullable = true)
 |-- School: integer (nullable = true)
 |-- 24X7Security: integer (nullable = true)
 |-- PowerBackup: integer (nullable = true)
 |-- CarParking: integer (nullable = true)
 |-- StaffQuarter: integer (nullable = true)
 |-- Cafeteria: integer (nullable = true)
 |-- MultipurposeRoom: integer (nullable = true)
 

### Vector Assembler

In [16]:
print("Assemble all features into a single feature vector column")
features_col = df_indexed.columns
features_col.remove('Price')
assembler = VectorAssembler(inputCols= features_col, 
                            outputCol= "features")
df_assembled = assembler.transform(df_indexed)

Assemble all features into a single feature vector column


In [17]:
print("Drop original individual feature columns after vectorisation")
df_assembled = df_assembled.drop(*features_col)
df_assembled.printSchema()

Drop original individual feature columns after vectorisation
root
 |-- Price: integer (nullable = true)
 |-- features: vector (nullable = true)



### Standard Scaler

In [18]:
print("Convert sparse vectors to dense format for scaling")
sparseToDense = F.udf(lambda v : Vectors.dense(v), VectorUDT())
df_assembled_dense = df_assembled.withColumn('features_array', sparseToDense('features'))

Convert sparse vectors to dense format for scaling


In [19]:
print("Preview dense vector transformation")
df_assembled_dense.select("features", "features_array").show(5, False)

Preview dense vector transformation
+---------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------+
|features                                                                                                             |features_array                                                                                                                                             |
+---------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------+
|(38,[0,1,2,15,16,17,27,29],[720.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])                                                      |[720.0,1.0,1.0,0.

In [20]:
print("View updated schema including the new dense vector column")
df_assembled_dense.printSchema()

View updated schema including the new dense vector column
root
 |-- Price: integer (nullable = true)
 |-- features: vector (nullable = true)
 |-- features_array: vector (nullable = true)



In [21]:
print("Drop original sparse 'features' column to retain only dense representation")
df_assembled_dense = df_assembled_dense.drop('features')
df_assembled_dense.printSchema()

Drop original sparse 'features' column to retain only dense representation
root
 |-- Price: integer (nullable = true)
 |-- features_array: vector (nullable = true)



In [22]:
print("Apply standard scaling to the dense features")
standardScaler = StandardScaler(inputCol="features_array", 
                                outputCol="features_scaled")

df_scaled = standardScaler.fit(df_assembled_dense).transform(df_assembled_dense)

Apply standard scaling to the dense features


In [23]:
print("Check schema after scaling")
df_scaled.printSchema()

Check schema after scaling
root
 |-- Price: integer (nullable = true)
 |-- features_array: vector (nullable = true)
 |-- features_scaled: vector (nullable = true)



In [24]:
print("Drop intermediate column to retain only final scaled features")
df_scaled = df_scaled.drop("features_array")
df_scaled.printSchema()

Drop intermediate column to retain only final scaled features
root
 |-- Price: integer (nullable = true)
 |-- features_scaled: vector (nullable = true)



In [25]:
print("Preview a sample record post-scaling")
df_scaled.show(1)

Preview a sample record post-scaling
+-------+--------------------+
|  Price|     features_scaled|
+-------+--------------------+
|4850000|[1.30679141107261...|
+-------+--------------------+
only showing top 1 row



### Train Test Split

In [26]:
print("Split dataset into training and testing sets (66.7% train, 33.3% test)")
train_df, test_df = df_scaled.randomSplit([0.667, 0.333], seed=0)

Split dataset into training and testing sets (66.7% train, 33.3% test)


In [27]:
print("Display record count in each split")
print("Observations in training set = ", train_df.count())
print("Observations in testing set = ", test_df.count())

Display record count in each split
Observations in training set =  5100
Observations in testing set =  2619


### Linear Regression

In [28]:
print("Fit a Linear Regression model with elastic net regularisation")
lr = LinearRegression(featuresCol = "features_scaled", 
                      labelCol = "Price", 
                      maxIter = 30, 
                      regParam = 0.3, 
                      elasticNetParam = 0.8)

lr_model = lr.fit(train_df)

Fit a Linear Regression model with elastic net regularisation


In [29]:
print("Generate predictions on the test set")
prediction_df = lr_model.transform(test_df)

Generate predictions on the test set


In [30]:
print("Display predicted vs actual values")
prediction_df.show(2, False)

Display predicted vs actual values
+-------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------+
|Price  |features_scaled                                                                                                                                                                                                                

In [31]:
print("Evaluate model performance using RMSE metric")
evaluator = RegressionEvaluator(labelCol="Price", 
                                predictionCol="prediction", 
                                metricName="rmse")

Evaluate model performance using RMSE metric


In [32]:
print("Compute RMSE for Linear Regression model")
rmse = evaluator.evaluate(prediction_df)
print("Root Mean Squared Error (RMSE) on test data =", rmse)

Compute RMSE for Linear Regression model
Root Mean Squared Error (RMSE) on test data = 18405061.603395633


#### Random Forest Regressor

In [33]:
print("Initialise Random Forest Regressor model")
rf = RandomForestRegressor(featuresCol="features_scaled", 
                           labelCol="Price")

Initialise Random Forest Regressor model


In [34]:
print("Define hyperparameter grid for tuning")
paramGrid = (ParamGridBuilder()
             .addGrid(rf.numTrees, [20, 50, 100])
             .addGrid(rf.maxDepth, [5, 10, 15])
             .addGrid(rf.maxBins, [32, 64])
             .build())

Define hyperparameter grid for tuning


In [35]:
print("Reuse the same evaluator for Random Forest model")
evaluator = RegressionEvaluator(labelCol="Price", 
                                predictionCol="prediction", 
                                metricName="rmse")

Reuse the same evaluator for Random Forest model


In [36]:
print("Setup 3-fold cross-validation")
crossval = CrossValidator(estimator=rf,
                         estimatorParamMaps=paramGrid,
                         evaluator=evaluator,
                         numFolds=3)

Setup 3-fold cross-validation


In [37]:
print("Train the model using cross-validation")
cv_model = crossval.fit(train_df)

Train the model using cross-validation


In [38]:
print("Generate predictions with the best model from cross-validation")
predictions = cv_model.transform(test_df)

Generate predictions with the best model from cross-validation


In [39]:
print("Evaluate and print RMSE for Random Forest model")
rmse = evaluator.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE) on test data = {rmse}")

Evaluate and print RMSE for Random Forest model
Root Mean Squared Error (RMSE) on test data = 17069091.874970634


In [40]:
# Closing spark session to release resources
spark.stop()